# Prerequisites
Sign up to https://ngrok.com/ to be able to reach Spark UI

In [1]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [2]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
        .appName('testColab') \
        .getOrCreate()

# Start a tunnel to access SparkUI

Open a ngrok tunnel to the HTTP server

In [4]:
from pyngrok import ngrok, conf
import getpass

print("Enter your authtoken, which can be copied "
"from https://dashboard.ngrok.com/get-started/your-authtoken")
conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken
··········


 * ngrok tunnel "https://c3b9-34-16-146-10.ngrok-free.app" -> "http://127.0.0.1:4040"


## Download Yellow Taxi Trip records data, read it in with Spark, and count the number of rows

Data source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [21]:
! wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-02 02:20:37--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.187, 18.155.128.222, 18.155.128.6, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  13.9MB/s    in 5.3s    

2025-03-02 02:20:43 (11.7 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [36]:
# from pyspark import SparkFiles

# file_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet'
# spark.sparkContext.addFile(file_url)

# df = spark.read.csv(SparkFiles.get('yellow_tripdata_2024-10.parquet'), header=True)
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2024-10.parquet')
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [39]:
# import pandas as pd
# from pyspark.sql import types

# df_pd = pd.read_parquet('yellow_tripdata_2024-10.parquet')
# df_pd.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-10-01 00:30:44,2024-10-01 00:48:26,1.0,3.0,1.0,N,162,246,1,18.4,1.0,0.5,1.5,0.0,1.0,24.9,2.5,0.0
1,1,2024-10-01 00:12:20,2024-10-01 00:25:25,1.0,2.2,1.0,N,48,236,1,14.2,3.5,0.5,3.8,0.0,1.0,23.0,2.5,0.0
2,1,2024-10-01 00:04:46,2024-10-01 00:13:52,1.0,2.7,1.0,N,142,24,1,13.5,3.5,0.5,3.7,0.0,1.0,22.2,2.5,0.0
3,1,2024-10-01 00:12:10,2024-10-01 00:23:01,1.0,3.1,1.0,N,233,75,1,14.2,3.5,0.5,2.0,0.0,1.0,21.2,2.5,0.0
4,1,2024-10-01 00:30:22,2024-10-01 00:30:39,1.0,0.0,1.0,N,262,262,3,3.0,3.5,0.5,0.0,0.0,1.0,8.0,2.5,0.0


In [ ]:
# Optionally put the tunnel down
# ngrok.disconnect(public_url)

# Homework

In [29]:
spark.version

'3.5.4'

In [30]:
df = df.repartition(4)
df.write.parquet('./partition', mode='overwrite')

In [31]:
! ls -lh ./partition/*.parquet

-rw-r--r-- 1 root root 24M Mar  2 02:26 ./partition/part-00000-2ea5d6dc-fb1a-450c-83bb-9703f2ec35d2-c000.snappy.parquet
-rw-r--r-- 1 root root 24M Mar  2 02:26 ./partition/part-00001-2ea5d6dc-fb1a-450c-83bb-9703f2ec35d2-c000.snappy.parquet
-rw-r--r-- 1 root root 24M Mar  2 02:26 ./partition/part-00002-2ea5d6dc-fb1a-450c-83bb-9703f2ec35d2-c000.snappy.parquet
-rw-r--r-- 1 root root 24M Mar  2 02:26 ./partition/part-00003-2ea5d6dc-fb1a-450c-83bb-9703f2ec35d2-c000.snappy.parquet


In [42]:
df_yellow = spark.read.parquet('./partition')
df_yellow.printSchema()
df_yellow.registerTempTable('yellow_taxi')

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [54]:
# df_yellow.show()
# df_yellow[(df_yellow.tpep_pickup_datetime >= '2024-10-15')&(df_yellow.tpep_pickup_datetime < '2024-10-16')].count()

128893

In [64]:
spark.sql("""
SELECT
    count(*) AS number_trips
FROM
    yellow_taxi
WHERE
    tpep_pickup_datetime >= '2024-10-15' AND tpep_pickup_datetime < '2024-10-16'
""").show()

+------------+
|number_trips|
+------------+
|      128893|
+------------+



In [71]:
spark.sql("""
SELECT
    max(timestampdiff(hour, tpep_pickup_datetime, tpep_dropoff_datetime)) AS max_trip_hours
FROM
    yellow_taxi
""").show()

+--------------+
|max_trip_hours|
+--------------+
|           162|
+--------------+



In [72]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-02 02:57:11--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.97, 52.85.39.117, 52.85.39.153, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.97|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-02 02:57:11 (139 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [95]:
df_zone = spark.read.csv('taxi_zone_lookup.csv', header=True)
df_zone.registerTempTable('taxi_zone')
df_zone.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [96]:
spark.sql(
    """
SELECT ID, Zone
FROM
  (SELECT
      PULocationID AS ID, COUNT(*) AS trip_count
  FROM
      yellow_taxi
  GROUP BY
      PULocationID
  ORDER BY
      trip_count ASC
  LIMIT 1) as fct
JOIN taxi_zone
ON taxi_zone.LocationID = fct.ID
""").show()

+---+--------------------+
| ID|                Zone|
+---+--------------------+
|105|Governor's Island...|
+---+--------------------+

